# Demo: Model Adaptability (lokales Modelltraining)

Kleiner End-to-End-Lauf der Methode aus Kapitel 3–4.

Ein Suchlauf weist drei Größen getrennt aus:

$$
\begin{aligned}
Q_{\mathrm{roh}}(S) &= \mathrm{AUC}_{S_{\mathrm{test}}}(f_{S_{\mathrm{train}}}) - \mathrm{AUC}_{S_{\mathrm{test}}}(f_{D_{\mathrm{train}}}) \\
Q_{\mathrm{gew}}(S) &= Q_{\mathrm{roh}}(S)\cdot \mathrm{cb}(S)^{\beta}\cdot (|S|/n)^{\alpha} \\
Q_{\mathrm{ga}}(S)  &= Q_{\mathrm{roh}}(S) - \max\bigl(\{0\}\cup\{Q_{\mathrm{roh}}(H)\mid H\text{ verallgemeinert }S\}\bigr)
\end{aligned}
$$

- `quality_raw` ist \(Q_{\mathrm{roh}}\) (Effektstärke).
- `quality` ist \(Q_{\mathrm{gew}}\) (Rangschlüssel; Protokoll \(\alpha=0{,}7\), \(\beta=0\)).
- `quality_ga` ist \(Q_{\mathrm{ga}}\) (Redundanzfilter, Rohskala).
- `interesting` ist das Erfolgskriterium: bewertbar, \(Q_{\mathrm{roh}}>0\), \(Q_{\mathrm{ga}}\ge 0\).

Zuerst ein Generator mit bekannter Struktur (`make_1_1_1`), danach derselbe Aufruf auf dem realen Datensatz **adult**. Die synthetische Suche ist sequentiell; adult nutzt `ProcessModelAdaptabilityDFS`. Adult setzt `data/adult.csv` voraus (`python experiments/experiments_real_world_data/download_data.py`).

In [1]:
from __future__ import annotations

import os

os.environ.setdefault("OMP_NUM_THREADS", "1")

import sys
from pathlib import Path

from sklearn.linear_model import LogisticRegression

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "tests" else HERE
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
SYNTH_DIR = EXPERIMENTS_DIR / "synthetical_data"

for path in (PROJECT_ROOT, SYNTH_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pysubgroup as ps
from datasets import make_1_1_1

print("pysubgroup:", ps.__file__)
print("project:", PROJECT_ROOT)
assert "pysubgroup/src/pysubgroup" in str(ps.__file__).replace("\\", "/"), (
    "Erwarte lokale Thesis-Installation von pysubgroup"
)

## 1. Synthetischer Datensatz

`a==0`: $y = -b + c$ · `a==1`: $y = b + c$ · Target = $(y > 0)$.

Das globale lineare Modell mittelt die gegensätzlichen Effekte; lokal pro `a`-Wert ist das Signal klar.

In [2]:
N = 2000
SEED = 42

spec = make_1_1_1(n=N, seed=SEED)
df = spec.df

print(spec.experiment_id, spec.name)
print(spec.description)
print("shape:", df.shape)
print("features F:", spec.feature_columns)
print("search S:", spec.search_columns)
print("ground truth:", spec.ground_truth_subgroups)
df.head()

sign_flip opposite_signal
Gegensätzliches Signal (Vorzeichenwechsel).
shape: (2000, 5)
features F: ['a', 'b', 'c']
search S: ['a']
ground truth: ['a==0', 'a==1']


,a,b,c,y,target
0,0,-0.875874,0.658442,1.534316,True
1,1,-0.083476,0.188192,0.104716,True
2,1,-0.741940,-0.510291,-1.252230,False
3,0,-0.695347,0.491350,1.186697,True
4,0,0.264566,-0.831038,-1.095604,False


## 2. Subgruppensuche (direkte API)

In [3]:
DEPTH = 1
MIN_SUPPORT = 20

qf = ps.LocalSoftClassifierPerformanceQF(
    model_builder_global=lambda: LogisticRegression(max_iter=1000, random_state=SEED),
    training_global=lambda m, X, y: m.fit(X, y),
    random_state=SEED,
)

target = ps.ModelAdaptabilityTarget(
    label_column=spec.label_column,
    feature_columns=spec.feature_columns,
)

search_space = ps.create_nominal_selectors_for_attribute(df, "a")

task = ps.ModelAdaptabilityDiscoveryTask(
    data=df,
    target=target,
    search_space=search_space,
    qf=qf,
    depth=DEPTH,
    result_set_size=10,
    constraints=[ps.MinSupportConstraint(MIN_SUPPORT)],
    generalization_awareness=True,
)

result = task.execute()
result_df = result.to_dataframe()
result_df

,quality_raw,quality,quality_ga,interesting,subgroup,size_sg,size_sg_train,size_sg_test,class_balance,global_test_score,local_test_score,local_train_score,status
0,0.000000,0.000000,0.000000,False,Dataset,2000,1000,1000,0.951220,0.822344,0.822344,0.839407,ok
1,0.223990,0.138075,0.223990,True,a==1,1002,503,499,0.923225,0.775993,0.999984,0.999794,ok
2,0.131371,0.080755,0.131371,True,a==0,998,497,501,0.980159,0.868581,0.999952,0.999968,ok


## 3. Realer Datensatz (`adult`)

In [ ]:
RW_DIR = EXPERIMENTS_DIR / "experiments_real_world_data"
sys.path.insert(0, str(RW_DIR))
# Zelle 1 hat bereits experiments/synthetical_data/datasets.py geladen.
for _mod in ("datasets", "models", "run", "preprocess", "plot_results"):
    sys.modules.pop(_mod, None)

from datasets import (
    feature_columns,
    prepare_dataset,
    stratified_split_fn,
)
from models import resolve_model_hooks
from run import build_search_space, default_min_support

assert ps.fork_available(), "ProcessModelAdaptabilityDFS braucht fork (Linux)"

DATASET = "adult"
df_rw, spec_rw = prepare_dataset(DATASET, seed=SEED)
ms_rw = default_min_support(len(df_rw))
fc_rw = feature_columns(df_rw)
split_rw = stratified_split_fn(seed=SEED, test_size=0.5)
train_rw, _ = split_rw(df_rw)
search_rw = build_search_space(train_rw, max(1, ms_rw // 2))

builder, train_g, train_l, pred_g, pred_l = resolve_model_hooks("lr", SEED)
qf_rw = ps.LocalSoftClassifierPerformanceQF(
    model_builder_global=builder,
    model_builder_local=builder,
    training_global=train_g,
    training_local=train_l,
    prediction_global=pred_g,
    prediction_local=pred_l,
    split_fn=split_rw,
    random_state=SEED,
)

task_rw = ps.ModelAdaptabilityDiscoveryTask(
    data=df_rw,
    target=ps.ModelAdaptabilityTarget(label_column="target", feature_columns=fc_rw),
    search_space=search_rw,
    qf=qf_rw,
    depth=2,
    result_set_size=-1,
    constraints=[ps.MinSupportConstraint(ms_rw)],
    generalization_awareness=True,
)

algo_rw = ps.ProcessModelAdaptabilityDFS(max_workers=min(8, os.cpu_count() or 1))
result_rw = task_rw.execute(algorithm=algo_rw)
result_rw_df = result_rw.to_dataframe()
result_rw_df

In [ ]:
print(result_rw_df.to_string())

In [ ]:
result_only_true = result_rw.to_dataframe(interesting_only=True)
print(result_only_true.to_string())
